In [12]:
import pandas as pd
import numpy as np
from anjana.anonymity import (
    k_anonymity_inner,
    k_anonymity,
    l_diversity,
    t_closeness,
    alpha_k_anonymity,
)
import masking_effects_on_xai_techniques.hierarchies as hi
from imblearn.under_sampling import RandomUnderSampler

In [62]:
path = "../data/adult/"

In [63]:
data = pd.read_csv(path + "data.csv")
len_data = len(data)

# Clean the data
data["income"] = data["income"].str.rstrip(".")
data.dropna(inplace=True)
data.drop(columns=["education-num", "fnlwgt"], inplace=True)

len_clean_data = len(data)
print(f"Dropped {len_data - len_clean_data} rows")

Dropped 1221 rows


In [64]:
data.shape

(47621, 13)

# Imbalance

In [65]:
df = data

In [66]:
df["income"].value_counts(normalize=True)

income
<=50K    0.757649
>50K     0.242351
Name: proportion, dtype: float64

In [67]:
# Assuming 'df' is your full DataFrame
X = df.drop("income", axis=1)  # Features (all columns except 'income')
y = df["income"]  # Target variable

In [68]:
minority_count = y.value_counts().min()

# Define the sampling strategy: set the majority count equal to the minority count
# y.value_counts().idxmax() gets the label of the majority class
sampling_strategy = {
    y.value_counts().idxmax(): minority_count,
    y.value_counts().idxmin(): minority_count,  # Minority count stays the same
}

# Initialize and apply the sampler
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

In [69]:
data = pd.concat([X_resampled, y_resampled], axis=1)

In [70]:
data.to_csv(path + "clean.csv", index=False)

# Hierarchies

In [31]:
hierarchies_path = "../hierarchies/adult"

for feat in ["age", "capital-gain", "capital-loss", "hours-per-week"]:
    try:
        hiear = hi.generate_qcut_hierarchy(data[feat], 7, search_for_n=True)
        hi.save_hierarchy(hiear, f"{hierarchies_path}/{feat}.csv")
    except RuntimeError:
        print("failed for feature ", feat)

failed for feature  hours-per-week


/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)


In [46]:
hiear = hi.generate_qcut_hierarchy(data["hours-per-week"], 3)  # , search_for_n=True)
hi.save_hierarchy(hiear, f"{hierarchies_path}/{feat}.csv")

/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)


In [47]:
hierarchies = {
    "age": dict(pd.read_csv(f"{hierarchies_path}/age.csv", header=None)),
    "education": dict(pd.read_csv(f"{hierarchies_path}/education.csv", header=None)),
    "marital-status": dict(
        pd.read_csv(f"{hierarchies_path}/marital-status.csv", header=None)
    ),
    "occupation": dict(pd.read_csv(f"{hierarchies_path}/occupation.csv", header=None)),
    "sex": dict(pd.read_csv(f"{hierarchies_path}/sex.csv", header=None)),
    "native-country": dict(
        pd.read_csv(f"{hierarchies_path}/native-country.csv", header=None)
    ),
    "workclass": dict(pd.read_csv(f"{hierarchies_path}/workclass.csv", header=None)),
    "relationship": dict(
        pd.read_csv(f"{hierarchies_path}/relationship.csv", header=None)
    ),
    "race": dict(pd.read_csv(f"{hierarchies_path}/race.csv", header=None)),
    "capital-gain": dict(
        pd.read_csv(f"{hierarchies_path}/capital-gain.csv", header=None)
    ),
    "capital-loss": dict(
        pd.read_csv(f"{hierarchies_path}/capital-loss.csv", header=None)
    ),
    "hours-per-week": dict(
        pd.read_csv(f"{hierarchies_path}/hours-per-week.csv", header=None)
    ),
}

In [41]:
quasi_ident = list(data.columns)
quasi_ident.remove("race")
quasi_ident.remove("income")

ident = [
    "race"
]  # Race is marked as an indentifier, because it is identical across so many instances
sens_att = "income"

In [48]:
all_counts_and_features = []
for feat, items in hierarchies.items():
    for item in items.values():
        length = len(item.unique())
        all_counts_and_features.append((length, feat))
sorted_data = sorted(all_counts_and_features, key=lambda x: x[0], reverse=True)
ordered_features = [feat for count, feat in sorted_data]

for feat, items in hierarchies.items():
    lengths = [f"{len(item.unique()):>4}" for item in items.values()]
    out = ", ".join(lengths)
    print(f"{feat:<15}: {out}")

print("Order in which features will be generalized:")
print(ordered_features)

age            :   72,    7,    6,    5,    4,    3,    2,    1
education      :   16,   11,    5,    3,    1
marital-status :    7,    2,    1
occupation     :   15,    7,    3,    1
sex            :    2,    1
native-country :   42,    7,    1
workclass      :    9,    4,    1
relationship   :    6,    1
race           :    5,    1
capital-gain   :  111,    7,    6,    5,    4,    3,    2,    1
capital-loss   :   83,    7,    6,    5,    4,    3,    2,    1
hours-per-week :   91,    3,    2,    1
Order in which features will be generalized:
['capital-gain', 'hours-per-week', 'capital-loss', 'age', 'native-country', 'education', 'occupation', 'education', 'workclass', 'age', 'marital-status', 'occupation', 'native-country', 'capital-gain', 'capital-loss', 'age', 'relationship', 'capital-gain', 'capital-loss', 'age', 'education', 'race', 'capital-gain', 'capital-loss', 'age', 'workclass', 'capital-gain', 'capital-loss', 'age', 'education', 'occupation', 'capital-gain', 'capital-loss', 

In [49]:
%%time
# Check with mina if this k is representative of used k in litterature
k = 10
supp_level = 20  # Select the suppression limit allowed
anon_df, supp_n, hiear = k_anonymity_inner(
    data, ident, quasi_ident, k, supp_level, hierarchies
)
max_supp_n = int(round(len(data) * supp_level / 100, 0))
print(f"Max rows that can be suppressed: {max_supp_n}")
print(f"Rows suppressed                : {supp_n}")
print(f"% of allowed rows suppressed   : {round(supp_n / max_supp_n * 100, 1)}%")
print(f"Generalization level           : {sum(hiear.values())}")
hiear

Max rows that can be suppressed: 4616
Rows suppressed                : 4569
% of allowed rows suppressed   : 99.0%
Generalization level           : 20
CPU times: user 7.83 s, sys: 35.9 ms, total: 7.87 s
Wall time: 7.86 s


{'age': 4,
 'workclass': 1,
 'education': 2,
 'marital-status': 1,
 'occupation': 2,
 'relationship': 1,
 'sex': 0,
 'capital-gain': 3,
 'capital-loss': 3,
 'hours-per-week': 1,
 'native-country': 2}

In [52]:
anon_df["income"].value_counts(normalize=True)

income
<=50K    0.553719
>50K     0.446281
Name: proportion, dtype: float64